# AI Product Discovery with RAG (Notebook Structure)

This notebook is the **orchestrator** for your repo-based implementation:

- `src/` holds core pipeline logic (cleaning, embeddings, FAISS, ranking, RAG, UI)
- `notebooks/` demonstrates the end-to-end workflow with plots + examples
- `data/` and `artifacts/` are **gitignored**

> **Colab tip**: If running in Colab, clone your repo and run from the repo root.


In [1]:
# Determine if running in Colab (robust, no import errors)
import sys, os

def in_colab() -> bool:
    # Safest: try importing google.colab (catch if package doesn't exist)
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        pass
    # Secondary: module already loaded
    if 'google.colab' in sys.modules:
        return True
    # Optional: environment hints
    return any(k in os.environ for k in ['COLAB_RELEASE_TAG','COLAB_GPU','COLAB_TPU_ADDR'])

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    # change the cwd to the notebooks
    os.chdir("/content/drive/MyDrive/pyramyd_ml")

    # Optional: !pip -q install -r requirements.txt
else:
    print('Running outside Colab')


Running outside Colab


In [ ]:
%%bash
export IN_COLAB=0
if python3 -c "import sys; import google.colab" &> /dev/null; then
  IN_COLAB=1
fi

if [ $IN_COLAB -eq 1 ]; then
    set -e
    # Mount Drive first (if not already)
    # python code runs before bash usually, so keep mount in Python cell above
    # Link persistent SSH config into runtime
    mkdir -p ~/.ssh
    ln -sf /content/drive/MyDrive/.ssh/id_ed25519 ~/.ssh/id_ed25519
    ln -sf /content/drive/MyDrive/.ssh/id_ed25519.pub ~/.ssh/id_ed25519.pub
    ln -sf /content/drive/MyDrive/.ssh/known_hosts ~/.ssh/known_hosts

    chmod 700 ~/.ssh
    chmod 600 ~/.ssh/id_ed25519
    chmod 644 ~/.ssh/id_ed25519.pub

    echo "SSH ready."

    git config --global user.name "Zhan"
    git config --global user.email "shizhe_zhang@berkeley.edu"
    ssh -T git@github.com || true
fi


In [2]:
# Ensure repo root is on sys.path so `import src.*` works
# !pwd
# !ls
from pathlib import Path

cwd = Path.cwd()

# If running from repo root, keep it; if from notebooks/, go up one level
if (cwd / 'src').exists():
    repo_root = cwd
elif (cwd.name == 'notebooks') and ((cwd.parent / 'src').exists()):
    repo_root = cwd.parent
    cwd = cwd.parent
else:
    # Fallback: search upwards for a folder containing `src` and `requirements.txt`
    candidates = [cwd, *cwd.parents]
    repo_root = next((p for p in candidates if (p / 'src').exists()), cwd)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', repo_root)
print('CWD:', cwd)
print('sys.path[0]:', sys.path[0])

Repo root: /Users/zhangshizhe/Dropbox/UCB/Pyramyd
CWD: /Users/zhangshizhe/Dropbox/UCB/Pyramyd
sys.path[0]: /Users/zhangshizhe/Dropbox/UCB/Pyramyd


In [3]:
# (Colab) If you cloned your repo, run this cell from the repo root.
# Example:
# !git clone git@github.com:<YOU>/<REPO>.git
# %cd <REPO>


# Install dependencies (Colab-only). Locally, use: pip install -r requirements.txt
# !pip -q install -r requirements.txt

import os, re, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repo modules
from src.config import EMBED_MODEL, LLM_MODEL, ARTIFACT_DIR
from src.data import load_data

# Uncomment as you implement these modules
# from src.embeddings import embed_texts
# from src.index import build_index
# from src.ranker import hybrid_score
# from src.rag import build_prompt

os.makedirs(ARTIFACT_DIR, exist_ok=True)
print("EMBED_MODEL:", EMBED_MODEL)
print("LLM_MODEL:", LLM_MODEL)
print("ARTIFACT_DIR:", ARTIFACT_DIR)


EMBED_MODEL: BAAI/bge-large-en-v1.5
LLM_MODEL: Qwen/Qwen2.5-7B-Instruct
ARTIFACT_DIR: artifacts


## 2) Load CSV + Inspect


In [4]:
# Path options:
# - In Colab: upload to /content, or mount Google Drive
# - Locally: point to your local file

CSV_PATH = "data/company_reviews.csv"

df_raw = load_data(os.path.join(cwd, CSV_PATH))

print("Shape:", df_raw.shape)
display(df_raw.head(3))

# Quick schema + missingness
col_info = pd.DataFrame({
    "col": df_raw.columns,
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "missing_%": [df_raw[c].isna().mean()*100 for c in df_raw.columns],
}).sort_values("missing_%", ascending=False)

display(col_info)


Shape: (17050, 20)


,name,rating,reviews,description,happiness,ceo_approval,ceo_count,ratings,locations,roles,salary,interview_experience,interview_difficulty,interview_duration,interview_count,headquarters,employees,industry,revenue,website
0,Sitel,NaN,NaN,"Sitel Group’s 75,000 people across the globe c...","{'Work Happiness Score': '55', 'Achievement': ...",70%,"CEO Approval is based on 4,612 ratings","{'Work/Life Balance': '3.4', 'Compensation/Ben...","{'Paradise, NV': '5.0', 'Pioneer, OH': '4.7', ...","{'Tier 1 Agent': '5.0', 'Director of Operation...",{'Customer Service Representative': '$14.48 pe...,Favorable,Easy,About a day or two,Based on 645 interviews,"600 Brickell Ave Miami, FL 33131 Vereinigte St...","10,000+",Telecommunications,$1B to $5B (USD),Twitter\nFacebook\nSitel website
1,Meadowbrook Rehabilitation,3.7,21 reviews,You'll work with the most experienced and loya...,{},NaN,NaN,"{'Work/Life Balance': '4.1', 'Compensation/Ben...",{},{},{},Favorable,Easy,NaN,Based on 5 interviews,Illinois,NaN,Healthcare,NaN,Meadowbrook Rehabilitation website
2,Intermountain,4.0,23 reviews,Why Intermountain?\n\nWe Bring Hope\n\nWith ou...,{},88%,CEO Approval is based on 17 ratings,"{'Work/Life Balance': '3.5', 'Compensation/Ben...",{},{},{'Mental Health Technician': '$13.16 per hour'...,Favorable,Medium,About a day or two,Based on 8 interviews,"Headquarters: 3240 Dredge Dr. Helena, MT 59602",201 to 500,Healthcare,$5M to $25M (USD),Twitter\nFacebook\nIntermountain website


,col,dtype,missing_%
18,revenue,object,40.310850
13,interview_duration,object,39.460411
5,ceo_approval,object,34.181818
6,ceo_count,object,34.181818
12,interview_difficulty,object,32.774194
14,interview_count,object,32.557185
11,interview_experience,object,32.557185
16,employees,object,11.888563
17,industry,object,10.826979
15,headquarters,object,10.674487


## 3) Cleaning & Parsing


In [ ]:
# Import reusable cleaners from src/ (source of truth)
from src.data import to_float, parse_listish, parse_salary

df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

# Numeric columns (edit as needed)
for c in ["rating", "happiness", "ceo_approval", "interview_difficulty", "interview_experience"]:
    if c in df.columns:
        df[c] = df[c].apply(to_float)

# List-like columns (edit as needed)
for c in ["roles", "locations", "ratings"]:
    if c in df.columns:
        df[c] = df[c].apply(parse_listish)

# Salary parsing
if "salary" in df.columns:
    parsed = df["salary"].apply(parse_salary)
    df["salary_min"] = [t[0] for t in parsed]
    df["salary_max"] = [t[1] for t in parsed]
    df["salary_median"] = [t[2] for t in parsed]

# Build doc text
text_cols = [c for c in ["description", "reviews"] if c in df.columns]
df["doc_text"] = df[text_cols].fillna("").astype(str).agg("\n".join, axis=1).str.replace(r"\s+", " ", regex=True).str.strip()

df_clean = df
print("Clean shape:", df_clean.shape)
display(df_clean.head(10))

Clean shape: (17050, 24)


,name,rating,reviews,description,happiness,ceo_approval,ceo_count,ratings,locations,roles,...,interview_count,headquarters,employees,industry,revenue,website,salary_min,salary_max,salary_median,doc_text
0,Sitel,NaN,NaN,"Sitel Group’s 75,000 people across the globe c...",NaN,70.0,"CEO Approval is based on 4,612 ratings","[{'Work/Life Balance': '3.4', 'Compensation/Be...","[{'Paradise, NV': '5.0', 'Pioneer, OH': '4.7',...","[{'Tier 1 Agent': '5.0', 'Director of Operatio...",...,Based on 645 interviews,"600 Brickell Ave Miami, FL 33131 Vereinigte St...","10,000+",Telecommunications,$1B to $5B (USD),Twitter\nFacebook\nSitel website,11.44,14.48,12.96,"Sitel Group’s 75,000 people across the globe c..."
1,Meadowbrook Rehabilitation,3.7,21 reviews,You'll work with the most experienced and loya...,NaN,NaN,NaN,"[{'Work/Life Balance': '4.1', 'Compensation/Be...",[{}],[{}],...,Based on 5 interviews,Illinois,NaN,Healthcare,NaN,Meadowbrook Rehabilitation website,NaN,NaN,NaN,You'll work with the most experienced and loya...
2,Intermountain,4.0,23 reviews,Why Intermountain?\n\nWe Bring Hope\n\nWith ou...,NaN,88.0,CEO Approval is based on 17 ratings,"[{'Work/Life Balance': '3.5', 'Compensation/Be...",[{}],[{}],...,Based on 8 interviews,"Headquarters: 3240 Dredge Dr. Helena, MT 59602",201 to 500,Healthcare,$5M to $25M (USD),Twitter\nFacebook\nIntermountain website,13.16,749.00,381.08,Why Intermountain? We Bring Hope With our holi...


## 4) Quick EDA


In [ ]:
# Keep EDA tight (4–6 plots max). Add/remove columns based on your data.

def hist_if_exists(col):
    if col in df_clean.columns and df_clean[col].notna().any():
        plt.figure()
        df_clean[col].dropna().plot(kind="hist", bins=30, title=f"Distribution: {col}")
        plt.xlabel(col)
        plt.show()

for col in ["rating", "happiness", "ceo_approval", "salary_median"]:
    hist_if_exists(col)

# Top roles / locations (optional)
def top_k_listcol(col, k=15):
    if col not in df_clean.columns:
        return None
    items = []
    for lst in df_clean[col]:
        items.extend(lst if isinstance(lst, list) else [])
    s = pd.Series(items).value_counts().head(k)
    plt.figure()
    s.sort_values().plot(kind="barh", title=f"Top {k}: {col}")
    plt.show()
    return s

_ = top_k_listcol("roles", k=15)
_ = top_k_listcol("locations", k=15)


## 5) Build Retrieval Documents


In [ ]:
# Define retrieval unit: start with 1 row -> 1 document.

NAME_COL = "name" if "name" in df_clean.columns else df_clean.columns[0]

def build_docs(df):
    docs = []
    for i, row in df.iterrows():
        meta = {"row_index": int(i), "company": str(row.get(NAME_COL, i))}
        # Keep a few structured signals for ranking/filters
        for c in ["rating","happiness","ceo_approval","salary_median","salary_min","salary_max","roles","locations"]:
            if c in df.columns:
                meta[c] = row.get(c)
        docs.append({"id": str(i), "text": str(row.get("doc_text","")), "meta": meta})
    return docs

docs = build_docs(df_clean)
print("Docs:", len(docs))
print("Example meta:", docs[0]["meta"])
print("Example text preview:", docs[0]["text"][:240])


## 6) Embeddings


In [ ]:
from src.config import EMBED_MODEL, ARTIFACT_DIR
from src.embeddings import embed_texts
import numpy as np
import os

# Ensure artifact dir exists
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# 1. Get list of text
# (Assuming 'docs' exists from Section 5. If not, run Section 5 first)
if 'docs' in locals():
    texts = [d["text"] for d in docs]
    print(f"Generating embeddings for {len(texts)} documents with {EMBED_MODEL}...")

    # 2. Embed
    emb = embed_texts(texts, EMBED_MODEL)

    # 3. Save
    save_path = os.path.join(ARTIFACT_DIR, "embeddings.npy")
    np.save(save_path, emb)
    print(f"Saved embeddings: {save_path} {emb.shape}")
else:
    print("Variable 'docs' not found. Please run Section 5 (Build Retrieval Documents) first.")

## 7) FAISS Index


In [ ]:
# FAISS setup + index build (do NOT generate src/index.py here)
import os, sys, subprocess

try:
    import faiss  # type: ignore
except Exception:
    # Colab often supports faiss-gpu; local installs often use faiss-cpu
    in_colab_env = any(k in os.environ for k in ["COLAB_RELEASE_TAG", "COLAB_GPU", "COLAB_TPU_ADDR"])
    pkg = "faiss-gpu" if in_colab_env else "faiss-cpu"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    import faiss  # type: ignore

from src.index import build_index
from src.config import ARTIFACT_DIR

os.makedirs(ARTIFACT_DIR, exist_ok=True)

if 'emb' in locals():
    print("Building FAISS index...")
    index = build_index(emb)

    index_path = os.path.join(ARTIFACT_DIR, "faiss.index")
    faiss.write_index(index, index_path)
    print(f"Saved index to {index_path}. Total vectors: {index.ntotal}")
else:
    print("Variable 'emb' not found. Please run Section 6 first.")

## 8) Retrieval + Filters Demo


In [ ]:
# Retrieval demo (do NOT generate src/retrieval.py here)
from src.retrieval import search

if 'index' in locals() and 'docs' in locals():
    query = "sharp eyes"
    print(f"\nQuery: {query}")
    results = search(query, index, docs, top_k=3)
    for i, r in enumerate(results, start=1):
        company = (r.get('meta') or {}).get('company', 'Unknown')
        print(f"[{i}] Score: {r['score']:.4f} | Company: {company}")
        print(f"    Text: {r['text'][:150]}...")
else:
    print("Skipping demo: 'index' or 'docs' not defined. Run Section 5 & 7 first.")

## 9) Hybrid Ranking with User Priorities


In [ ]:
# Hybrid ranking demo (do NOT generate src/ranker.py here)
from src.ranker import hybrid_score

if 'results' in locals() and results:
    print("\n--- Re-ranking by Salary priority ---")
    priorities = {"salary": 0.5, "rating": 0.1}
    ranked = hybrid_score(results, priorities)
    for i, r in enumerate(ranked, start=1):
        sal = (r.get('meta') or {}).get('salary_median', 'N/A')
        company = (r.get('meta') or {}).get('company', 'Unknown')
        print(f"[{i}] Hybrid: {r['hybrid_score']:.4f} (Sem: {r['score']:.4f}) | Salary: {sal} | {company}")
else:
    print("Skipping ranking demo: 'results' not defined. Run retrieval section first.")

## 10) RAG Answer Generation (Grounded)


In [ ]:
# RAG demo (do NOT generate src/rag.py here)
from src.rag import ask_rag

if 'model' in locals() and 'tokenizer' in locals() and 'ranked' in locals():
    query = locals().get('query', "sharp eyes")
    print("\n--- Generative Answer ---")
    answer = ask_rag(query, ranked[:3], model, tokenizer)
    print(answer)
else:
    print("Model or results not ready. Please ensure ranking + LLM setup are complete.")

## 11) Side-by-Side Comparison


In [ ]:
# TODO:
# - Select top N or user-selected companies
# - Create a structured comparison table
# - Ask LLM for a comparison summary (salary-first vs culture/WLB-first) with citations

raise NotImplementedError("Implement comparison utilities (suggest: src/compare.py), then run comparison here.")


## 12) Lightweight Evaluation


In [ ]:
# Lightweight evaluation demo (uses src/eval.py)
from collections import defaultdict
from src.eval import EvalQuery, evaluate_retrieval, summarize

# Weak supervision example: query by company name and treat all rows of that company as relevant.
# This is imperfect, but gives a simple Recall@K signal without manual labeling.
if 'docs' in locals() and 'index' in locals():
    # Build company -> ids map
    company_to_ids = defaultdict(set)
    for d in docs:
        company = (d.get('meta') or {}).get('company')
        if company:
            company_to_ids[str(company)].add(str(d.get('id')))

    # Pick a few companies with multiple docs (adjust N as needed)
    candidates = [c for c, ids in company_to_ids.items() if len(ids) >= 2]
    sample_companies = candidates[:5]
    eval_set = [EvalQuery(query=c, relevant_ids=frozenset(company_to_ids[c])) for c in sample_companies]

    if not eval_set:
        print("No suitable companies found for weak-supervision eval.")
    else:
        rows = evaluate_retrieval(index=index, docs=docs, queries=eval_set, k=10)
        for r in rows:
            print(f"query={r.query!r} recall@{r.k}={r.recall_at_k:.3f} latency_ms={r.latency_ms:.1f}")
        print("\nSummary:", summarize(rows))
else:
    print("Run retrieval/index sections first so 'docs' and 'index' exist.")

## 13) Gradio Demo UI


In [ ]:
# Gradio demo UI (uses src/ui.py)
from src.ui import build_app

# Set to True if you want to launch the UI from the notebook.
LAUNCH_UI = False

demo = build_app()
print("UI built. Set LAUNCH_UI=True to launch.")
if LAUNCH_UI:
    demo.launch(share=True)

## 14) Save Artifacts + Local Run Notes


In [ ]:
# Optional artifact bundling:
# import shutil
# shutil.make_archive("artifacts_bundle", "zip", ARTIFACT_DIR)
# print("Created artifacts_bundle.zip")


## 15) Resume Bullets & Next Steps

- Built a RAG-based product discovery engine with FAISS semantic search over company reviews + metadata.
- Designed a hybrid ranking model combining embedding similarity with structured signals and user-weighted priorities.
- Deployed an interactive Gradio UI enabling natural-language search, filtering, and AI side-by-side comparisons with citations.


In [ ]:
# LLM utilities live in src/llm.py (do NOT overwrite source files from the notebook).
from src.config import LLM_MODEL
from src.llm import load_qwen_model, generate_response, generate_chat

print(f"LLM helpers imported. Model configured: {LLM_MODEL}")
print("Next cells will load the model and generate text (may download weights).")

In [ ]:
from src.config import LLM_MODEL
from src.llm import load_qwen_model, generate_response

# 1. Load the model (loads from local cache on Drive)
model, tokenizer = load_qwen_model(LLM_MODEL)

# 2. Generate a response
prompt = "tell a story about a cute piggy"
response = generate_response(model, tokenizer, prompt)

print("\nResponse:")
print(response)

In [ ]:
prompt = "tell a story about a cute piggy"
response = generate_response(model, tokenizer, prompt)

print("\nResponse:")
print(response)

In [ ]:
# Install sentence-transformers if missing (Colab/local friendly)
import sys, subprocess

try:
    import sentence_transformers  # type: ignore
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    import sentence_transformers  # type: ignore

print("sentence-transformers ready")

In [ ]:
# Embeddings sanity check (do NOT overwrite src/config.py or src/embeddings.py here)
from src.config import EMBED_MODEL
from src.embeddings import embed_texts

test_docs = [
    "The L4 GPU is great for AI inference.",
    "Qwen is a powerful open-source LLM.",
    "RAG combines retrieval with generation.",
]

print(f"\nTesting embedding generation with {EMBED_MODEL}...")
emb_test = embed_texts(test_docs, EMBED_MODEL)
print(f"Success! Output shape: {emb_test.shape}")
print(f"Sample (first 3 dims): {emb_test[0][:3]}")